In [17]:
import pandas as pd
import joblib
import gdown
from tqdm import tqdm
from nltk.corpus import wordnet as wn
import numpy as np

def get_synsets(word):
    return wn.synsets(word)

def get_antonyms(word):
    antonyms = set()
    for syn in wn.synsets(word):
        for lem in syn.lemmas():
            for ant in lem.antonyms():
                antonyms.add(ant.name())
    return antonyms

def semantic_features(X):
    features = []
    for pair in tqdm(X, desc="Extracting semantic features"):
        premise, hypothesis = pair.split(" [SEP] ")
        p_words = set(premise.split())
        h_words = set(hypothesis.split())

        syn_overlap = sum(
            1 for hw in h_words if get_synsets(hw)
            and any(get_synsets(hw)[0].lowest_common_hypernyms(get_synsets(pw)[0])
                    for pw in p_words if get_synsets(pw))
        )

        antonyms = sum(
            1 for hw in h_words for pw in p_words
            if get_antonyms(hw) and hw != pw
        )

        features.append([syn_overlap, antonyms])
    return np.array(features)

In [18]:
url = f"https://drive.google.com/uc?id=1ilMvFGJQyKflsDgTvnxl7OTtt4ISq-wj"
model_file = "nli_svm_model.joblib"

gdown.download(url, output=model_file, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1ilMvFGJQyKflsDgTvnxl7OTtt4ISq-wj
To: /kaggle/working/nli_svm_model.joblib
100%|██████████| 11.5M/11.5M [00:00<00:00, 212MB/s]


'nli_svm_model.joblib'

In [ ]:
test_data_path = "../data/test_data/test.csv"

In [ ]:
pipeline = joblib.load(model_file)

df_test = pd.read_csv(test_data_path)
X_test_raw = df_test[["premise", "hypothesis"]]

In [20]:
preds = pipeline.predict(X_test_raw)

Extracting semantic features: 100%|██████████| 3302/3302 [00:08<00:00, 404.76it/s]


In [21]:
output = pd.DataFrame({"prediction": preds})
output.to_csv("Group_52_A.csv", index=False)